# Phase 1A - Short-term memory (01-05)

**Phase 1A - Short-term memory (01-05)** - an independent notebook (runnable standalone in Colab or locally).

Covers: Short-term memory (01-05) - keep recent turns.

Attribution: adapted from *Agent Memory Techniques* by Nir Diamant (https://github.com/NirDiamant/Agent_Memory_Techniques), Apache-2.0. Inline demos are original.

## 0. Setup (self-contained - run this first)

In [ ]:
# Self-contained setup - works standalone in Google Colab or locally.
import sys, os, subprocess
from pathlib import Path
REPO_URL = "https://github.com/syaikhipin/kdd26-memdiag"
try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
if IN_COLAB:
    repo = Path("/content/kdd26-memdiag")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], check=False)
    SOURCE = repo / "experiment" / "github_submission" / "source"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib", "pyyaml"], check=False)
else:
    SOURCE = None
    for cand in [Path.cwd(), *Path.cwd().parents]:
        for sub in ("source", "experiment"):
            if (cand / sub / "run.py").exists():
                SOURCE = cand / sub
                break
        if SOURCE:
            break
    if SOURCE is None:
        raise FileNotFoundError("Run from the repo root (or in Colab it auto-clones).")
sys.path.insert(0, str(SOURCE))
os.environ.setdefault("OPENAI_BASE_URL", "https://api.openai.com/v1")
SOURCE_DIR = SOURCE
PROJECT_ROOT = SOURCE.parent
RESULTS_DIR = PROJECT_ROOT / "results"
print("SOURCE_DIR =", SOURCE, "| IN_COLAB =", IN_COLAB)

from embedder import TfidfHashEmbedder
from llm_client import LLMConfig, make_client, OfflineLLMClient
from memory_core import MemoryRecord
from providers import build_provider
emb = TfidfHashEmbedder()
try:
    llm = make_client(LLMConfig(backend='openai-compatible' if os.environ.get('OPENAI_API_KEY') else 'offline',
                                base_url=os.environ.get('OPENAI_BASE_URL','https://api.openai.com/v1'),
                                model='gpt-4o'))
except Exception:
    llm = OfflineLLMClient()
print('LLM backend:', llm.backend, '| API key:', bool(os.environ.get('OPENAI_API_KEY')))
TURNS = [
 ("Alice", "Hi, I am Alice. I work as a data scientist at a health-tech startup in Berlin."),
 ("Bob", "I am Bob, an ML engineer in Athens. I prefer PyTorch."),
 ("Alice", "We deploy on Kubernetes and track runs with Weights and Biases."),
 ("Bob", "Our training run failed last night with CUDA OOM at batch 256."),
 ("Alice", "We hit that before. Reducing batch to 64 and enabling gradient checkpointing fixed it."),
 ("Bob", "Our best val_loss was 0.423 with lr=3e-4 and weight_decay=0.01."),
 ("Alice", "I live in Prenzlauer Berg. My favorite coffee shop is on Kollwitzplatz."),
 ("Bob", "Let us sync next Tuesday at 10am CET."),
]
records = [MemoryRecord(f"t{i}", f"{w}: {t}", {"session_id": "s1" if i < 4 else "s2"}) for i, (w, t) in enumerate(TURNS)]
print("setup OK | turns =", len(TURNS))


## Short-term memory (01-05) - keep recent turns

**01 - Conversation Buffer** · Save the full conversation verbatim

In [ ]:
# Technique 01 - Conversation Buffer: Save the full conversation verbatim
print('01 buffer keeps all', len(TURNS), 'turns verbatim')

**02 - Sliding Window** · Keep only the last few messages

In [ ]:
# Technique 02 - Sliding Window: Keep only the last few messages
print('02 sliding window keeps last 4:', [w for w,_ in TURNS[-4:]])

**03 - Summary Memory** · Replace old turns with a short summary

In [ ]:
# Technique 03 - Summary Memory: Replace old turns with a short summary
try:
 _s = llm.summarize(chr(10).join(f'{w}: {t}' for w,t in TURNS[:5]))
 print('03 summary (LLM):', _s[:80])
except Exception:
 print('03 summary (offline): Alice & Bob discussed ML work; an OOM was fixed by batch 64.')

**04 - Summary Buffer** · Summarize old turns, keep recent verbatim

In [ ]:
# Technique 04 - Summary Buffer: Summarize old turns, keep recent verbatim
print('04 summary+buffer: summary of old + keep recent:', [w for w,_ in TURNS[-2:]])

**05 - Token Buffer** · Trim history to a token budget

In [ ]:
# Technique 05 - Token Buffer: Trim history to a token budget
b=40; out=[]; used=0
for w,t in reversed(TURNS):
 n=len(t.split())
 if used+n>b: break
 out.append(w); used+=n
print('05 token budget(40) keeps:', list(reversed(out)))